# Demo 07 — Held-out test evaluation

This notebook measures the adapter trained by Demo 06 on the **held-out test split**, which played no part in training or epoch selection.

**What you will learn**

* why a fine-tuned model must be compared with a *fair* baseline: the base model given the same instructions;
* how to read category, severity, joint, and valid-JSON metrics, and the delta between two models.

**What is compared**

| Column | Model | System prompt |
| --- | --- | --- |
| Base model | Original Qwen3-1.7B | The prompt the adapter was trained with |
| Fine-tuned adapter | Qwen3-1.7B + Demo 06 adapter | The prompt the adapter was trained with |

Both models receive **the same system prompt**, so the delta column measures what fine-tuning adds, not what the prompt adds.

> **Lesson from specification 021.** Without a system prompt the base model does not know the task, the JSON format, or the label codes, and it never answers with valid JSON: a baseline without a prompt mostly measures the output format and overstates the gain of fine-tuning (51 points instead of about 22 on the earlier data). This notebook therefore measures the base model only with the adapter's prompt.

In [1]:
from llm_fine_tuning.device import describe_runtime, require_bfloat16_mps_device
from llm_fine_tuning.display import show_key_values, show_prediction, show_table
from llm_fine_tuning.models import load_adapter_model, load_base_model, load_tokenizer, release_model_memory
from llm_fine_tuning.paths import require_adapter_directory, validate_model_directory
from llm_fine_tuning.tickets.data import TicketProjectPaths, load_test_split
from llm_fine_tuning.tickets.evaluation import evaluate_model, generate_responses, metrics_rows
from llm_fine_tuning.tickets.prompts import adapter_run_name, load_system_prompt, verify_adapter_prompt

## 1. Settings

| Setting | Meaning |
| --- | --- |
| `ADAPTER_PROMPT_VARIANT` | The `prompt_variant` used in Demo 06; it selects the adapter directory and the prompt |
| `TEST_GENERATION_BATCH_SIZE` | Test tickets classified together; lower it if MPS memory is short |
| `GENERATION_MAX_NEW_TOKENS` | Longest answer the model may write |
| `TEST_RECORD_INDEX` | Test ticket shown in the final inspection cell (0 to the number of test records minus 1; 0 to 209 with the current split) |

In [2]:
ADAPTER_PROMPT_VARIANT = "short"  # must match Demo 06: 'none' (A), 'short' (B), or 'full' (C)
TEST_GENERATION_BATCH_SIZE = 4
GENERATION_MAX_NEW_TOKENS = 128
TEST_RECORD_INDEX = 3

## 2. Check the environment, the adapter, and its prompt

The adapter directory is derived from the prompt variant. `prompt_variant.json`, written by Demo 06, must record the same variant and exactly the same prompt text: evaluating an adapter with a prompt it was not trained with would give misleading results. The same prompt is then given to the base model.

In [3]:
paths = TicketProjectPaths.discover()
adapter_directory = paths.models_directory / adapter_run_name(ADAPTER_PROMPT_VARIANT) / "adapter"
validate_model_directory(paths.model_directory)
require_adapter_directory(adapter_directory)

system_prompt = load_system_prompt(paths.prompts_directory, ADAPTER_PROMPT_VARIANT)
verify_adapter_prompt(adapter_directory, ADAPTER_PROMPT_VARIANT, system_prompt)
device = require_bfloat16_mps_device()

show_key_values("Runtime", describe_runtime(device) | {"Adapter directory": adapter_directory, "Prompt variant (both models)": ADAPTER_PROMPT_VARIANT})

| Runtime | Value |
| --- | ---: |
| `macOS` | `26.7.1` |
| `Machine` | `arm64` |
| `Python` | `3.11.0` |
| `PyTorch` | `2.14.1` |
| MPS available | `True` |
| Selected device | `mps` |
| Adapter directory | `/Users/lsaetta/Progetti/llm-fine-tuning-on-mac/ticket-classification/artifacts/models/demo06-qwen3-1.7b-ticket-classification-lora-prompt-short/adapter` |
| Prompt variant (both models) | `short` |

## 3. Load the held-out test split

Only `test_dataset_v2.jsonl` is loaded, and it is validated with the same rules as the training data.

In [4]:
test_split = load_test_split(paths)
print(f"Validated held-out test records: {len(test_split)}")

Validated held-out test records: 210


## 4. Measure the base model

The original model classifies every test ticket once, with the adapter's prompt. Generation is deterministic (no sampling), so a rerun gives the same answers. The model is then deleted to free MPS memory before the adapter is loaded.

In [5]:
tokenizer = load_tokenizer(paths.model_directory)
base_model = load_base_model(paths.model_directory, device)
base_column = f"Base model, prompt '{ADAPTER_PROMPT_VARIANT}'"
results = {}
results[base_column], _ = evaluate_model(
    base_model, tokenizer, test_split, system_prompt, device,
    TEST_GENERATION_BATCH_SIZE, GENERATION_MAX_NEW_TOKENS, f"Evaluating {base_column}",
)
del base_model
release_model_memory()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating Base model, prompt 'short':   0%|          | 0/210 [00:00<?, ?ticket/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


## 5. Measure the fine-tuned adapter and compare

An answer that is not valid JSON counts as wrong for every label metric.

**How to read the table.** Compare each metric with a trivial reference: always answering the most frequent test severity (P3) is right 27% of the time (57 of 210 tickets), and a random category 10% of the time. Differences of about 3 points or less are within sampling noise on 210 tickets.

In [6]:
adapter_model = load_adapter_model(load_base_model(paths.model_directory, device), adapter_directory, device)
adapter_column = f"Fine-tuned adapter, prompt '{ADAPTER_PROMPT_VARIANT}'"
results[adapter_column], _ = evaluate_model(
    adapter_model, tokenizer, test_split, system_prompt, device,
    TEST_GENERATION_BATCH_SIZE, GENERATION_MAX_NEW_TOKENS, f"Evaluating {adapter_column}",
)
show_table(["Metric", *results, "Delta (fine-tuned - base, same prompt)"], metrics_rows(results, delta=(adapter_column, base_column)))

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating Fine-tuned adapter, prompt 'short':   0%|          | 0/210 [00:00<?, ?ticket/s]

| Metric | Base model, prompt 'short' | Fine-tuned adapter, prompt 'short' | Delta (fine-tuned - base, same prompt) |
| --- | ---: | ---: | ---: |
| Category Accuracy | 66.2% | 99.0% | +32.9% |
| Severity Accuracy | 67.1% | 95.7% | +28.6% |
| Category+Severity Accuracy | 43.3% | 94.8% | +51.4% |
| Valid JSON Rate | 100.0% | 100.0% | +0.0% |

## 6. Inspect one test prediction

Change `TEST_RECORD_INDEX` in the settings cell, rerun that cell, then rerun this one to compare the ticket, the adapter's answer, and the expected answer. This is a visual check only; it does not change the metrics.

In [7]:
if not 0 <= TEST_RECORD_INDEX < len(test_split):
    raise IndexError(f"TEST_RECORD_INDEX must be between 0 and {len(test_split) - 1}.")
record = test_split[TEST_RECORD_INDEX]
response = generate_responses(adapter_model, tokenizer, [record], system_prompt, device, 1, GENERATION_MAX_NEW_TOKENS)[0].strip()
show_prediction(f"Test record {TEST_RECORD_INDEX}", record["messages"][0]["content"], response, record["messages"][1]["content"])

### Test record 3

**Ticket**
```text
The database cluster endpoint cannot be accessed by clients. Within the local setup, the issue is low impact and can be handled during normal support work.
```

**Fine-tuned response is valid JSON:** `True`

**Fine-tuned response**
```json
{"category":"DB-02","severity":"P4","summary":"Database cluster endpoint is unreachable"}
```

**Expected response**
```json
{"category":"DB-02","severity":"P4","summary":"Database cluster endpoint is unavailable"}
```

## Next steps

* Record the table in the specification of the experiment you are running.
* Before committing, clear the notebook outputs: `jupyter nbconvert --clear-output --inplace <notebook>`.